In [1]:
import pandas as pd

df = pd.read_excel("../data/raw/tourism_2021_2025.xlsx", sheet_name="G1", header=None)
df.head(15)

,0,1,2,3,4,5,6,7,8,9
0,(Last update: 12-04-2026),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021 data was revised based on revised populat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,تمت مراجعة بيانات 2021 بناءً على تعداد السكان ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,G1,Key Tourism Indicators in the Kingdom,NaN,NaN,NaN,NaN,NaN,NaN,NaN,المؤشرات الرئيسة للسياحة في المملكة
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,Unit,2021.000000,2022.000000,2023.000000,2024.000000,2025.000000,الوحدة,NaN
8,NaN,Inbound Tourism,NaN,NaN,NaN,NaN,NaN,NaN,NaN,السياحة الوافدة
9,NaN,Tourists Number (Overnight Visitors),('000),3477.216000,16637.861000,27423.536000,29727.443000,29254.787000,('000),عدد السياح (زوار المبيت)


In [2]:
# 1) ناخذ الصفوف من 8 وتحت، والأعمدة من 1 لين 7
g1 = df.iloc[8:, 1:8]

# 2) نعطي الأعمدة أسماء واضحة
g1.columns = ["indicator", "unit", 2021, 2022, 2023, 2024, 2025]

# 3) نعرف كل صف لأي نوع سياحة يتبع
g1["tourism_type"] = g1["indicator"].where(g1["unit"].isna())
g1["tourism_type"] = g1["tourism_type"].ffill()

# 4) نحذف صفوف العناوين (اللي ما لها وحدة)
g1 = g1.dropna(subset=["unit"])

g1

,indicator,unit,2021,2022,2023,2024,2025,tourism_type
9,Tourists Number (Overnight Visitors),('000),3477.216000,16637.861000,27423.536000,2.972744e+04,2.925479e+04,Inbound Tourism
10,Overnight Stays,('000),31771.381000,270728.182000,432299.345000,5.602272e+05,5.476839e+05,Inbound Tourism
11,Tourists Spending,SAR Mn,14715.547390,98294.585215,141233.978818,1.685242e+05,1.766287e+05,Inbound Tourism
12,Average Length of Stay,Night,9.137016,16.271815,15.763808,1.884546e+01,1.872117e+01,Inbound Tourism
13,Avergae Sending per Trip,SAR,4231.990000,5907.890000,5150.100000,5.668980e+03,6.037600e+03,Inbound Tourism
14,Average Spending per Night,SAR,463.170000,363.070000,326.700000,3.008100e+02,3.225000e+02,Inbound Tourism
16,Tourists Number (Overnight Visitors),('000),63833.898000,77836.501000,81920.362000,8.615736e+04,9.332351e+04,Domestic Tourism
17,Overnight Stays,('000),353331.203000,369606.393000,495341.184000,5.386176e+05,5.640685e+05,Domestic Tourism
18,Tourists Spending,SAR Mn,102184.113023,107259.216265,114406.733550,1.152758e+05,1.270765e+05,Domestic Tourism
19,Average Length of Stay,Night,5.535166,4.748497,6.046619,6.251556e+00,6.044227e+00,Domestic Tourism


In [3]:
# 1) نقلب الجدول: السنوات تنزل من أعمدة إلى صفوف
g1_long = g1.melt(
    id_vars=["tourism_type", "indicator", "unit"],
    var_name="year",
    value_name="value",
)

# 2) نرتّب النصوص
g1_long["indicator"] = g1_long["indicator"].str.strip()
g1_long["indicator"] = g1_long["indicator"].str.replace("Avergae Sending", "Average Spending")
g1_long["tourism_type"] = g1_long["tourism_type"].str.replace(" Tourism", "")

# 3) السنة تصير رقم صحيح
g1_long["year"] = g1_long["year"].astype(int)

print(g1_long.shape)
g1_long.head(10)

(90, 5)


,tourism_type,indicator,unit,year,value
0,Inbound,Tourists Number (Overnight Visitors),('000),2021,3477.216000
1,Inbound,Overnight Stays,('000),2021,31771.381000
2,Inbound,Tourists Spending,SAR Mn,2021,14715.547390
3,Inbound,Average Length of Stay,Night,2021,9.137016
4,Inbound,Average Spending per Trip,SAR,2021,4231.990000
5,Inbound,Average Spending per Night,SAR,2021,463.170000
6,Domestic,Tourists Number (Overnight Visitors),('000),2021,63833.898000
7,Domestic,Overnight Stays,('000),2021,353331.203000
8,Domestic,Tourists Spending,SAR Mn,2021,102184.113023
9,Domestic,Average Length of Stay,Night,2021,5.535166


In [4]:
import os

# نسوي مجلد للبيانات النظيفة (إذا موجود ما يسوي شي)
os.makedirs("../data/clean", exist_ok=True)

# نحفظ الجدول كملف CSV
g1_long.to_csv("../data/clean/kingdom_yearly.csv", index=False, encoding="utf-8-sig")

print("تم الحفظ ✅")

تم الحفظ ✅


In [5]:
# نشيل المسافات الزايدة من الوحدة
g1_long["unit"] = g1_long["unit"].str.strip()

# نتأكد: لازم يطلع 4 وحدات بس
print(g1_long["unit"].unique())

# نعيد الحفظ
g1_long.to_csv("../data/clean/kingdom_yearly.csv", index=False, encoding="utf-8-sig")
print("تم الحفظ ✅")


<StringArray>
['('000)', 'SAR Mn', 'Night', 'SAR']
Length: 4, dtype: str
تم الحفظ ✅


In [6]:
# نقرأ ورقة المناطق
df2 = pd.read_excel("../data/raw/tourism_2021_2025.xlsx", sheet_name="G2", header=None)

g2 = df2.iloc[8:, 1:8]
g2.columns = ["indicator", "unit", 2021, 2022, 2023, 2024, 2025]
g2["indicator"] = g2["indicator"].str.strip()

# صفوف العناوين: نوعين
is_header = g2["unit"].isna()                                  # أي عنوان
is_region = is_header & g2["indicator"].str.endswith("PROVINCE")  # عنوان منطقة

g2["region"] = g2["indicator"].where(is_region).ffill()
g2["tourism_type"] = g2["indicator"].where(is_header & ~is_region).ffill()

g2 = g2.dropna(subset=["unit"])

# نقلبه للشكل الطويل
g2_long = g2.melt(
    id_vars=["region", "tourism_type", "indicator", "unit"],
    var_name="year",
    value_name="value",
)

# نرتّب النصوص
g2_long["indicator"] = g2_long["indicator"].str.replace("Avergae", "Average").str.replace("Sending", "Spending")
g2_long["tourism_type"] = g2_long["tourism_type"].str.replace(" Tourism", "")
g2_long["region"] = g2_long["region"].str.replace(" PROVINCE", "").str.title()
g2_long["unit"] = g2_long["unit"].str.strip()
g2_long["year"] = g2_long["year"].astype(int)

print(g2_long.shape)
print(g2_long["region"].unique())
print(g2_long["indicator"].unique())

(1170, 6)
<StringArray>
[    'Baha',   'Qassim',    'Assir',  'Eastern',     'Hail',    'Jazan',
     'Jouf',  'Madinah',   'Makkah',   'Najran', 'Northern',   'Riyadh',
    'Tabuk']
Length: 13, dtype: str
<StringArray>
['Tourists Number (Overnight Visitors)',
                      'Overnight Stays',
                    'Tourists Spending',
               'Average Length of Stay',
            'Average Spending per Trip',
           'Average Spending per Night']
Length: 6, dtype: str


In [7]:
# مجموع السياح من كل المناطق
tourists = g2_long[g2_long["indicator"] == "Tourists Number (Overnight Visitors)"]
regions_total = tourists.groupby(["tourism_type", "year"])["value"].sum()

# نفس الرقم من جدول المملكة
kingdom = g1_long[g1_long["indicator"] == "Tourists Number (Overnight Visitors)"]
kingdom_total = kingdom.set_index(["tourism_type", "year"])["value"]

# نقارن
check = pd.DataFrame({"regions_sum": regions_total, "kingdom": kingdom_total})
check["difference"] = check["regions_sum"] - check["kingdom"]
print(check.round(1))

# نحفظ
g2_long.to_csv("../data/clean/regions_yearly.csv", index=False, encoding="utf-8-sig")
print("تم الحفظ ✅")

                   regions_sum   kingdom  difference
tourism_type year                                   
Domestic     2021      63833.9   63833.9         0.0
             2022      77836.5   77836.5         0.0
             2023      81920.4   81920.4         0.0
             2024      86157.4   86157.4         0.0
             2025      93323.5   93323.5        -0.0
Inbound      2021       3477.2    3477.2         0.0
             2022      16637.9   16637.9        -0.0
             2023      27423.5   27423.5         0.0
             2024      29727.4   29727.4        -0.0
             2025      29254.8   29254.8         0.0
Internal     2021      67311.1   67311.1         0.0
             2022      94474.4   94474.4         0.0
             2023     109343.9  109343.9         0.0
             2024     115884.8  115884.8         0.0
             2025     122578.3  122578.3         0.0
تم الحفظ ✅


In [8]:
MONTHS = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

def clean_monthly(sheet, tourism_type):
    raw = pd.read_excel("../data/raw/tourism_2021_2025.xlsx", sheet_name=sheet, header=None)

    # ناخذ الأعمدة من 1 لين 8 ونسمّيها
    t = raw.iloc[:, 1:9]
    t.columns = ["year", "month", "tourists_000", "nights_000", "spending_sar_mn",
                 "avg_stay_nights", "avg_spend_per_trip_sar", "avg_spend_per_night_sar"]

    # السنة مكتوبة بس عند يناير، فنعبّي الباقي
    t["year"] = pd.to_numeric(t["year"], errors="coerce").ffill()

    # نخلي بس صفوف الشهور (يروح العنوان وصف Total)
    t["month"] = t["month"].astype(str).str.strip()
    t = t[t["month"].isin(MONTHS)]

    t["year"] = t["year"].astype(int)
    t["month_num"] = t["month"].map({m: i + 1 for i, m in enumerate(MONTHS)})
    t["tourism_type"] = tourism_type

    # نحوّل الأرقام لأرقام فعلية
    num_cols = ["tourists_000", "nights_000", "spending_sar_mn",
                "avg_stay_nights", "avg_spend_per_trip_sar", "avg_spend_per_night_sar"]
    t[num_cols] = t[num_cols].apply(pd.to_numeric)
    return t

# نستخدم الدالة مرتين ونجمع النتيجتين في جدول واحد
monthly = pd.concat([clean_monthly("I1", "Inbound"),
                     clean_monthly("D1", "Domestic")], ignore_index=True)

print(monthly.shape)
monthly.head()

(120, 10)


,year,month,tourists_000,nights_000,spending_sar_mn,avg_stay_nights,avg_spend_per_trip_sar,avg_spend_per_night_sar,month_num,tourism_type
0,2021,January,232.765,2342.151,947.841324,10.062301,4072.10,404.69,1,Inbound
1,2021,February,125.032,1250.771,582.029154,10.003604,4655.04,465.34,2,Inbound
2,2021,March,148.294,1136.270,502.989244,7.662279,3391.84,442.67,3,Inbound
3,2021,April,176.031,1393.891,800.808055,7.918431,4549.24,574.51,4,Inbound
4,2021,May,184.845,1186.067,744.948248,6.416545,4030.12,628.08,5,Inbound


In [9]:
# نجمع الشهور لكل سنة
monthly_sum = monthly.groupby(["tourism_type", "year"])[["tourists_000", "spending_sar_mn"]].sum()

# نجيب نفس الأرقام من G1
def from_g1(indicator):
    rows = g1_long[g1_long["indicator"] == indicator]
    return rows.set_index(["tourism_type", "year"])["value"]

check = monthly_sum.copy()
check["tourists_diff"] = check["tourists_000"] - from_g1("Tourists Number (Overnight Visitors)")
check["spending_diff"] = check["spending_sar_mn"] - from_g1("Tourists Spending")
print(check.round(1))

# نحفظ
monthly.to_csv("../data/clean/monthly.csv", index=False, encoding="utf-8-sig")
print("تم الحفظ ✅")

                   tourists_000  spending_sar_mn  tourists_diff  spending_diff
tourism_type year                                                             
Domestic     2021       63833.9         102184.1            0.0            0.0
             2022       77836.5         107259.2            0.0            0.0
             2023       81920.4         114406.7            0.0            0.0
             2024       86157.4         115275.8            0.0            0.0
             2025       93323.5         127076.5            0.0            0.0
Inbound      2021        3477.2          14715.5            0.0           -0.0
             2022       16637.9          98294.6            0.0            0.0
             2023       27423.5         141234.0            0.0            0.0
             2024       29727.4         168524.2            0.0            0.0
             2025       29254.8         176628.7            0.0            0.0
تم الحفظ ✅


In [10]:
def clean_breakdown(sheet, categories, category_name):
    raw = pd.read_excel("../data/raw/tourism_2021_2025.xlsx", sheet_name=sheet, header=None)

    # ناخذ: السنة + الشهر + أعمدة الفئات
    t = raw.iloc[:, 1:3 + len(categories)]
    t.columns = ["year", "month"] + categories

    # نفس خطوات الشهري
    t["year"] = pd.to_numeric(t["year"], errors="coerce").ffill()
    t["month"] = t["month"].astype(str).str.strip()
    t = t[t["month"].isin(MONTHS)]

    # الفئات تنزل من أعمدة إلى صفوف
    long = t.melt(id_vars=["year", "month"], var_name=category_name, value_name="tourists_000")
    long["year"] = long["year"].astype(int)
    long["month_num"] = long["month"].map({m: i + 1 for i, m in enumerate(MONTHS)})
    long["tourists_000"] = pd.to_numeric(long["tourists_000"])
    return long

purpose = clean_breakdown(
    "I2",
    ["Religious", "Business", "Leisure", "Other", "Visiting Friends & Relatives"],
    "purpose",
)
origin = clean_breakdown(
    "I3",
    ["GCC", "Middle East", "Africa", "Asia", "Europe", "Americas", "Other"],
    "origin_region",
)

# تحقق: المجموع لازم يطابق عدد الوافدين لكل سنة
print(purpose.groupby("year")["tourists_000"].sum().round(1))
print(origin.groupby("year")["tourists_000"].sum().round(1))

purpose.to_csv("../data/clean/inbound_purpose.csv", index=False, encoding="utf-8-sig")
origin.to_csv("../data/clean/inbound_origin.csv", index=False, encoding="utf-8-sig")
print("تم الحفظ ✅")

year
2021     3477.2
2022    16637.9
2023    27424.0
2024    29727.4
2025    29254.8
Name: tourists_000, dtype: float64
year
2021     3477.2
2022    16637.9
2023    27423.5
2024    29727.4
2025    29254.8
Name: tourists_000, dtype: float64
تم الحفظ ✅


In [11]:
raw4 = pd.read_excel("../data/raw/tourism_2021_2025.xlsx", sheet_name="I4", header=None)

top10 = raw4.iloc[:, 1:6]
top10.columns = ["year", "rank", "country", "tourists_000", "market_share"]

top10["year"] = pd.to_numeric(top10["year"], errors="coerce").ffill()

# نخلي بس الصفوف اللي فيها ترتيب (1 لين 10)
top10["rank"] = pd.to_numeric(top10["rank"], errors="coerce")
top10 = top10.dropna(subset=["rank"])

top10["year"] = top10["year"].astype(int)
top10["rank"] = top10["rank"].astype(int)
top10["country"] = top10["country"].str.strip()
top10[["tourists_000", "market_share"]] = top10[["tourists_000", "market_share"]].apply(pd.to_numeric)

print(top10.shape)
top10.to_csv("../data/clean/inbound_top10_countries.csv", index=False, encoding="utf-8-sig")
print("تم الحفظ ✅")

top10[top10["year"] == 2025]

(50, 5)
تم الحفظ ✅


,year,rank,country,tourists_000,market_share
58,2025,1,Egypt,3174.258,0.108504
59,2025,2,Pakistan,2805.110,0.095886
60,2025,3,India,2035.652,0.069584
61,2025,4,Indonesia,2020.524,0.069066
62,2025,5,Kuwait,1836.573,0.062779
63,2025,6,Bahrain,1440.868,0.049252
64,2025,7,Turkey,1165.875,0.039852
65,2025,8,Yemen,1165.221,0.039830
66,2025,9,Iraq,1110.364,0.037955
67,2025,10,UAE,1100.142,0.037606


In [12]:
top10[top10["year"] == 2021]

,year,rank,country,tourists_000,market_share
6,2021,1,Kuwait,623.462,0.179299
7,2021,2,India,316.488,0.091018
8,2021,3,Egypt,308.253,0.088649
9,2021,4,Pakistan,257.640,0.074094
10,2021,5,Qatar,254.013,0.073051
11,2021,6,Bahrain,239.947,0.069005
12,2021,7,Jordan,196.898,0.056625
13,2021,8,Syria,160.372,0.046121
14,2021,9,Sudan,153.844,0.044243
15,2021,10,Yemen,109.923,0.031612


In [ ]:
baha = g2_long[(g2_long["region"] == "Baha") & (g2_long["tourism_type"] == "Inbound")]
baha.pivot(index="indicator", columns="year", values="value").round(1)  

year,2021,2022,2023,2024,2025
indicator,,,,,
Average Length of Stay,7.7,9.9,6.8,6.7,60.1
Average Spending per Night,575.5,216.2,392.6,410.0,105.2
Average Spending per Trip,4416.2,2132.8,2663.9,2734.1,6318.9
Overnight Stays,5.3,41.1,110.3,54.1,349.8
Tourists Number (Overnight Visitors),0.7,4.2,16.3,8.1,5.8
Tourists Spending,3.1,8.9,43.3,22.2,36.8


: 